# B02. Watching the interpreter stop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/cpython-internals/blob/main/lessons/b02-the-debugger/b02.ipynb)

Every lesson so far has watched Python from inside Python. You printed a code object, you counted references, you walked the frame chain. All of it was the interpreter describing itself while it was running.

This lesson stops it instead.

![the eight stages of the pipeline with the last one highlighted](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/b02-the-debugger/diagrams/where-we-are.svg)

Stopping a program in the middle and looking at it is the oldest tool there is, and it comes in two halves. One half you already have and can run right now. The other half needs a debug build and a C debugger, so it was run for you and written down, command by command, and you can read it without installing anything.

## About the source references

Now and then this lesson points at CPython's own source, like this: `Lib/pdb.py:488@v3.15.0rc1#Pdb`.

Read it as four parts: the file, the lines, the release those line numbers belong to, and the name of the thing they are inside.

Every reference is a link, and every one is checked against the pinned source on each change, so a stale reference fails the build instead of sending you somewhere wrong. You never have to read any of it. The references are there so you can go deeper when you want to, and so you can check that this lesson is not making things up.

## Setup

Colab does not come with the small package these lessons use, so the next cell installs it. If you are running this from a checkout of the repository it is already installed and the cell does nothing.

In [ ]:
import sys

if sys.version_info < (3, 14):
    print("This lesson needs CPython 3.14 or newer.")
    print(f"This runtime is {sys.version.split()[0]}, and the cells below will not run on it.")
else:
    try:
        import pyxray
    except ImportError:
        %pip install -q "pyxray @ git+https://github.com/tamnd/cpython-internals@main#subdirectory=pyxray"
        import pyxray

## Which Python is this

Everything below was checked against the version this cell prints and against 3.14. Where the two disagree, the lesson says so.

In [ ]:
import pyxray

pyxray.show()

## The debugger you already have

`pdb` ships with Python and has since 1992. Most people meet it by typing `breakpoint()`, getting a `(Pdb)` prompt, and then not knowing what to type next.

The prompt is the problem. A debugging session at a prompt is something you perform once and then cannot show anybody, which is why so much debugging knowledge is passed around as folklore.

So do it the other way round. pdb reads its commands from whatever you hand it as standard input, so a debugging session can be written down in advance, run, and read back later, and the session below is a list of six strings. The class takes the streams as arguments: [Lib/pdb.py:488@v3.15.0rc1#Pdb](https://github.com/python/cpython/blob/v3.15.0rc1/Lib/pdb.py#L488).

The program is four functions deep and ends in one multiplication, which is the same program the gdb sessions further down use.

In [ ]:
import io
import pdb
from pathlib import Path

source = """def double(n):
    return n * 2


def middle(n):
    return double(n)


def top():
    return middle(21)


print(top())
"""

program = Path("program.py")
program.write_text(source)

commands = ["break double", "continue", "where", "args", "list", "continue"]

typed = io.StringIO("\n".join(commands) + "\n")
printed = io.StringIO()

session = pdb.Pdb(stdin=typed, stdout=printed, readrc=False)
session.run(compile(source, str(program), "exec"), {"__name__": "__main__"})

print(printed.getvalue())

program.unlink()

> **Version note.** The path in front of every line number is wherever this notebook is running, so it is different for everybody. The line numbers themselves are not.

Read that as six answers rather than as a wall of text.

`break double` picks a function. `continue` runs until it is called. `where` prints the call stack, innermost first, and there are four Python frames in it because there were four Python calls. `args` prints the arguments of the frame you are stopped in, and `n` is 21. `list` shows the source around the stopping point, with `B->` marking the line the breakpoint is on.

That is the whole vocabulary. Six commands, and you have just done a real debugging session and can hand it to somebody else as a file.

## What a debugger actually is

A debugger sounds like it must be doing something special. It is not. pdb is ordinary Python that asks the interpreter to call it back on every function call, line and return, which is a hook any program can install, and the hook is one function call: [Lib/bdb.py:232-236@v3.15.0rc1#start_trace](https://github.com/python/cpython/blob/v3.15.0rc1/Lib/bdb.py#L232-L236).

Here is a debugger in a handful of lines. It does one thing, which is print the calls it was asked to watch, and there is nothing missing from it except features.

In [ ]:
import sys

# The hook fires on every call anywhere in the process, and in a notebook that includes
# Jupyter's own, so this only prints the three functions the cell is about.
WATCHING = {"top", "middle", "double"}


def watch(frame, event, arg):
    if frame.f_code.co_qualname in WATCHING:
        print(f"call    {frame.f_code.co_qualname:8}  line {frame.f_lineno}")
    return None


def double(n):
    return n * 2


def middle(n):
    return double(n)


def top():
    return middle(21)


sys.settrace(watch)
top()
sys.settrace(None)

Three calls, in the order they happened. `pdb` is that with a prompt attached, a list of breakpoints, and the good sense to return itself so it keeps getting called.

The name check is the only thing in there that is not about debugging. A trace hook is global: once it is installed the interpreter calls it for every function anywhere, so without the check you would also watch whatever Jupyter does between one line of this cell and the next.

In 3.14 and later, pdb prefers [sys.monitoring](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#monitoring-events) when it can get it and falls back to `sys.settrace` otherwise, which is the `if` in the lines cited above. The idea is the same either way: the interpreter offers to tell you when things happen, and a debugger is whatever you do with that offer.

## What none of this can see

Everything above lives inside the interpreter, which is exactly why it stops where it does.

![a table of four debugging tools and what each one can see](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/b02-the-debugger/diagrams/four-ways-to-look.svg)

`pdb` can tell you that `double` called `n * 2`. It cannot tell you what happened next, because what happened next was C. It also cannot help you at all if the process dies outright, because a dead process has no Python left in it to run a debugger written in Python.

For those two questions you need a debugger that works on the process rather than in it. That means [gdb](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#gdb), and gdb means a debug build, and a debug build means either twenty minutes with a compiler or the container from B01.

## Somebody already ran it

So it was run for you.

Both sessions below were run in the debug image this project publishes, pinned by digest, and every line gdb printed was written down and committed under `debugger/`. They are checked the same way the rest of this material is: a job re runs both sessions in the same image and compares them line by line, with addresses allowed to move and nothing else. If a future CPython changes the shape of the stack, that job goes red and this lesson gets rewritten.

You do not need any of it to read what follows. If you want to run it yourself, `just build-gdb` in a checkout will, and the exercises at the end say how to do it by hand.

## Two stacks, one moment

The program is deliberately dull. Four functions, one multiplication, one `print`.

```python
# Four Python calls deep, ending in one multiplication.
#
# The multiplication is the point. It is the last thing this program does before it has an
# answer, so a breakpoint on the C function behind * stops the interpreter at a moment we can
# describe from both sides.
#
# Four functions, one operator, one print, and no docstring. The docstring is left out on
# purpose: it would land in the module's globals, and gdb prints the globals dictionary in
# every frame that carries one.


def double(n):
    return n * 2


def middle(n):
    return double(n)


def top():
    return middle(21)


print(top())
```

The multiplication is the target. `n * 2` in Python is `PyNumber_Multiply` in C, [Objects/abstract.c:1176@v3.15.0rc1#PyNumber_Multiply](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/abstract.c#L1176), and stopping there stops the interpreter at a moment that can be described from both sides at once.

Getting there takes four commands, and the first one is not obvious.

**1. `tbreak pymain_run_file`**

The interpreter runs a lot of Python of its own before it reaches your file, so stop once at the moment it is about to start. The t means this fires only once.

```text
Temporary breakpoint 1 at 0x3601a0: file Modules/main.c, line 447.
```

**2. `run /tmp/program.py`**

Start the program under the debugger. It stops at the temporary breakpoint.

```text
[Thread debugging using libthread_db enabled]
Using host libthread_db library "/lib/aarch64-linux-gnu/libthread_db.so.1".

Temporary breakpoint 1, pymain_run_file (config=config@entry=0xaaaaab197898 <_PyRuntime+144048>) at Modules/main.c:447
447	{
```

**3. `break PyNumber_Multiply`**

Now that startup is out of the way, break on the C function behind the * operator.

```text
Breakpoint 2 at 0xaaaaaabb7a38: file Objects/abstract.c, line 1177.
```

**4. `continue`**

Let it run until the multiplication. The two arguments printed are the operands.

```text
Breakpoint 2, PyNumber_Multiply (v=v@entry=21, w=w@entry=2) at Objects/abstract.c:1177
1177	{
```

The temporary breakpoint on `pymain_run_file` is there because of something worth knowing on its own. By the time the interpreter reaches your file it has already run a great deal of Python, most of it `site.py` setting up the import system, and a lot of that Python multiplies things. Break on `PyNumber_Multiply` first and you stop inside `posixpath.dirname` before your program has started. So the session stops once at the point the interpreter is about to run your file, [Modules/main.c:446@v3.15.0rc1#pymain_run_file](https://github.com/python/cpython/blob/v3.15.0rc1/Modules/main.c#L446), and only then sets the real breakpoint.

Now the interpreter is frozen, holding 21 and 2, and there are two ways to ask it what it is doing.

**5. `bt -frame-arguments none`**

The whole C stack, with the argument values left out, because a CPython frame carries whole dictionaries and printing them buries the shape. Count the _PyEval_EvalFrameDefault frames.

```text
#0  PyNumber_Multiply (v=..., w=...) at Objects/abstract.c:1177
#1  0x0000aaaaaad2ed04 in _PyEval_EvalFrameDefault (tstate=..., frame=..., throwflag=...) at Python/generated_cases.c.h:65
#2  0x0000aaaaaad4b624 in _PyEval_EvalFrame (tstate=..., frame=..., throwflag=...) at ./Include/internal/pycore_ceval.h:122
#3  0x0000aaaaaad4b7b8 in _PyEval_Vector (tstate=..., func=..., locals=..., args=..., argcount=..., kwnames=...) at Python/ceval.c:2153
#4  0x0000aaaaaad4b890 in PyEval_EvalCode (co=..., globals=..., locals=...) at Python/ceval.c:688
#5  0x0000aaaaaadd10c0 in run_eval_code_obj (tstate=..., co=..., globals=..., locals=...) at Python/pythonrun.c:1398
#6  0x0000aaaaaadd1704 in run_mod (mod=..., filename=..., globals=..., locals=..., flags=..., arena=..., interactive_src=..., generate_new_source=...) at Python/pythonrun.c:1501
#7  0x0000aaaaaadd2004 in _PyRun_File (fp=..., filename=..., start=..., globals=..., locals=..., closeit=..., flags=...) at Python/pythonrun.c:1324
#8  0x0000aaaaaadd2dcc in _PyRun_SimpleFile (fp=..., filename=..., closeit=..., flags=...) at Python/pythonrun.c:536
#9  0x0000aaaaaadd3a3c in _PyRun_AnyFile (fp=..., filename=..., closeit=..., flags=...) at Python/pythonrun.c:84
#10 0x0000aaaaaae000d4 in pymain_run_file_obj (program_name=..., filename=..., skip_source_first_line=...) at Modules/main.c:437
#11 0x0000aaaaaae001e8 in pymain_run_file (config=...) at Modules/main.c:458
#12 0x0000aaaaaae00754 in pymain_run_python (exitcode=...) at Modules/main.c:750
#13 0x0000aaaaaae00978 in Py_RunMain () at Modules/main.c:837
#14 0x0000aaaaaae009dc in pymain_main (args=...) at Modules/main.c:867
#15 0x0000aaaaaae00a5c in Py_BytesMain (argc=..., argv=...) at Modules/main.c:891
#16 0x0000aaaaaab3dff4 in main (argc=..., argv=...) at ./Programs/python.c:15
```

Seventeen frames. Read from the bottom: `main` calls `Py_BytesMain` calls `pymain_main`, down through the file running machinery, into `PyEval_EvalCode`, and then into the eval loop.

Now count the eval loop frames. There is exactly one.

![seventeen C frames beside four Python frames](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/b02-the-debugger/diagrams/two-stacks.svg)

four nested Python calls run inside a single _PyEval_EvalFrameDefault frame, so the C stack does not grow one frame per Python call. That is T07's whole point, and here it is as a number you can count rather than a claim you have to take.

The other side of the same instant looks like this.

**6. `py-bt`**

The same instant, described in Python. Count these frames and compare.

```text
Traceback (most recent call first):
  File "/tmp/program.py", line 13, in double
    return n * 2
  File "/tmp/program.py", line 17, in middle
    return double(n)
  File "/tmp/program.py", line 21, in top
    return middle(21)
  File "/tmp/program.py", line 24, in <module>
    print(top())
```

Four frames, and they are the four functions in the program. This is the same chain you walked from the inside in T07 by following `f_back`, except that here nothing in the program is cooperating. The process is stopped and gdb is reading its memory.

Two answers to one question, both true. C says seventeen, Python says four, and neither is a simplification of the other. The C stack is where the interpreter is. The Python stack is what the interpreter is doing.

The last two commands go one level further down, into a single object.

**7. `print ((PyObject *) v)->ob_type->tp_name`**

v is the left operand. Every object starts with a pointer to its type, and this reads the name out of memory rather than asking type() for it.

```text
$1 = 0xaaaaaaeee328 "int"
```

**8. `print ((PyObject *) v)->ob_refcnt`**

The other half of the object header. This number is worth staring at.

```text
$2 = 3221225472
```

`v` is the left operand, the 21. Every Python object starts with a header, and the first thing in that header is a pointer to its type, so `((PyObject *) v)->ob_type->tp_name` reads the string `"int"` straight out of memory. Nothing asked the object what it was. The answer was already lying there.

The second number is the good one. `3221225472` is not a count of anything. It is 3 times 2 to the 30, which CPython uses to mark an object as [immortal](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#immortal-object): [Include/refcount.h:47@v3.15.0rc1#_Py_IMMORTAL_INITIAL_REFCNT](https://github.com/python/cpython/blob/v3.15.0rc1/Include/refcount.h#L47).

small integers report a reference count of 3221225472, which is a marker meaning never free this rather than a count of references, and you can check the same number from a browser tab right now.

In [ ]:
import sys

print("sys.getrefcount(21) =", sys.getrefcount(21))
print("3 << 30            =", 3 << 30)
print("the same number:", sys.getrefcount(21) == 3 << 30)

The same value, from two completely different places. One came out of a stopped process on another machine with a C expression, the other came from a function call in your browser. That agreement is the point of this lesson: the debugger is not showing you a different Python, it is showing you the same one from further back.

## Where py-bt comes from

`py-bt` looks like magic and is not. gdb loads a Python script that CPython ships in its own tree, [Tools/gdb/libpython.py:2122-2131@v3.15.0rc1#PyBacktrace](https://github.com/python/cpython/blob/v3.15.0rc1/Tools/gdb/libpython.py#L2122-L2131), and that script does what you would do by hand.

![five steps from the C stack to a Python filename and line number](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/b02-the-debugger/diagrams/where-py-bt-comes-from.svg)

It scans the C stack for frames named `_PyEval_EvalFrameDefault`, [Tools/gdb/libpython.py:1778-1791@v3.15.0rc1#is_evalframe](https://github.com/python/cpython/blob/v3.15.0rc1/Tools/gdb/libpython.py#L1778-L1791), reads the `frame` argument out of each one, and follows it into a `_PyInterpreterFrame`, [Include/internal/pycore_interpframe_structs.h:29-53@v3.15.0rc1#_PyInterpreterFrame](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_interpframe_structs.h#L29-L53). From there `f_executable` is the code object, and the code object has the filename, the function name and the table that turns an instruction offset into a line number.

Five pointer hops. Somebody wrote them down once, and now everybody gets `py-bt`.

## Where a crash came from

The second session is about the case pdb cannot reach at all.

![an exception compared with a segfault](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/b02-the-debugger/diagrams/two-ways-to-stop.svg)

An exception is a thing the interpreter builds and hands back to you, with a traceback that names the file and the line. A [segmentation fault](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#segmentation-fault) is not. The kernel takes the process away mid instruction, and there is no interpreter left to explain anything. Nothing prints. The shell reports 139 and you are on your own.

Pure Python does not usually get you there, which is why the program below has to work at it. `ctypes` is the door out of the language, and past that door a mistake is a crash rather than an exception. Extension modules live on the other side of the same door, so this is a small version of what a bug in one looks like from outside.

```python
# Read one byte from address zero, on purpose.
#
# There is no way to do this by accident in Python, which is the point. ctypes is the door out
# of the language, and past that door a mistake is a segfault rather than an exception.
# Extension modules live on the other side of the same door, so this is a small version of
# what a bug in one looks like from the outside.
#
# This program prints nothing. It dies three calls down, and working out which three is the
# whole exercise.

import ctypes


def read_nothing():
    return ctypes.string_at(0)


def ask():
    return read_nothing()


print(ask())
```

**1. `run /tmp/program.py`**

No breakpoints this time. Run it and wait for it to fall over.

```text
[Thread debugging using libthread_db enabled]
Using host libthread_db library "/lib/aarch64-linux-gnu/libthread_db.so.1".

Program received signal SIGSEGV, Segmentation fault.
0x0000fffff7df4110 in ?? () from /lib/aarch64-linux-gnu/libc.so.6
```

**2. `bt 4`**

The top four C frames. The first is inside the C library and has no name, which is about as much as a C stack alone was ever going to tell you.

```text
#0  0x0000fffff7df4110 in ?? () from /lib/aarch64-linux-gnu/libc.so.6
#1  0x0000fffff795c708 in string_at (ptr=0x0, size=-1) at ./Modules/_ctypes/_ctypes.c:6103
#2  0x0000fffff7926a94 in ?? () from /lib/aarch64-linux-gnu/libffi.so.8
#3  0x0000fffff7926088 [PAC] in ?? () from /lib/aarch64-linux-gnu/libffi.so.8
```

`SIGSEGV` and an address, and then a C stack whose top frame does not even have a name, because it is inside the C library where there is nothing to unwind with. That is roughly what a crash report from a C extension looks like when it reaches you, and it is not much to go on.

Then one command.

**3. `py-bt`**

The same crash, in Python. This is the answer, and it took one command.

```text
Traceback (most recent call first):
  File "/opt/python/lib/python3.15/ctypes/__init__.py", line 610, in string_at
    return _string_at(ptr, size)
  File "/tmp/program.py", line 15, in read_nothing
    return ctypes.string_at(0)
  File "/tmp/program.py", line 19, in ask
    return read_nothing()
  File "/tmp/program.py", line 22, in <module>
    print(ask())
```

**4. `py-list`**

The Python source around the line that did it, read out of the stopped process.

```text
 605    _string_at = PYFUNCTYPE(py_object, c_void_p, c_int)(_string_at_addr)
 606    def string_at(ptr, size=-1):
 607        """string_at(ptr[, size]) -> string
 608    
 609        Return the byte string at void *ptr."""
>610        return _string_at(ptr, size)
 611    
 612    _memoryview_at = PYFUNCTYPE(
 613        py_object, c_void_p, c_ssize_t, c_int)(_memoryview_at_addr)
 614    def memoryview_at(ptr, size, readonly=False):
 615        """memoryview_at(ptr, size[, readonly]) -> memoryview
```

A filename, a line number and a function name, out of a process that is no longer running. `py-list` then reads the Python source around that line out of the same stopped process.

That is the whole reason this material bothers with a debug build. Everything else in these twelve lessons could be done from a browser tab. This could not, and it is also the thing you will actually want on the worst day of a project.

## Try it yourself

**One.** Change the command list in the pdb cell. Try `step` instead of `continue`, or add `p n * 2` after `args` to evaluate an expression at the stopping point. Every command is documented in the `pdb` module docs.

**Two.** Put `print(frame.f_locals)` inside `watch` in the four line tracer and run it again. You now have a tool that prints every argument of every call in your program, which is about fifty lines short of a profiler.

**Three.** Return `watch` from `watch` instead of `None` and see how much more it prints. That one line is the difference between tracing calls and tracing lines, and it is why real tracers are careful about what they return.

**Four.** Run the two stacks session yourself: `docker run --rm -it --cap-add=SYS_PTRACE --security-opt seccomp=unconfined ghcr.io/tamnd/cpython-internals/cpython:debug gdb -q /usr/local/bin/python3`. Write the program to a file first, and try `py-locals` and `py-up`, which this lesson did not use.

**Five.** Take the crash program and wrap the `ctypes` call in `try` and `except Exception`. Run it again and watch the crash go straight through the handler, because there is no exception involved and nothing for it to catch.

## What just happened

You have a debugger already, it takes its commands from a list as happily as from a keyboard, and a session written down is a session you can share.

pdb is not special. It is a callback the interpreter offers to anybody, and a four line version of it fits in a cell.

A stopped interpreter has two stacks, and both are honest. Seventeen C frames and four Python frames described the same instant, with one `_PyEval_EvalFrameDefault` covering all four Python calls.

The object header is right there in memory. The type name and the reference count of the number 21 came out of a C expression, and one of those numbers is a marker meaning immortal rather than a count, which you checked yourself from a browser.

`py-bt` is five pointer hops that somebody wrote down once, in a file CPython ships in its own tree.

And when a process dies with no traceback at all, that same file gets you the Python line that did it.

## Where this goes next

B03 is the test suite, which is the other thing a build gives you. CPython ships about seven hundred thousand lines of tests, and knowing how to run one of them against a change you made is what turns reading this material into being able to alter CPython.